# Global Superstore Data Cleaning & Preparation

This notebook focuses on cleaning and preparing the Global Superstore dataset using Python and Pandas.

The objective is to identify and handle missing values, check for duplicate records, validate data types and values, clean text data, perform data quality checks, and export the cleaned dataset for further analysis.

In [ ]:
import pandas as pd

In [ ]:
%pip install xlrd

## 1. Load the Dataset

The Global Superstore dataset is loaded from the raw data folder using Pandas.

In [ ]:
file_path = "../data/raw/Global Superstore.xls"

df = pd.read_excel(file_path)

df.head()

## 2. Initial Data Inspection

The dataset is inspected to understand its size, column names, data types, and overall structure.

In [ ]:
# (number of rows, number of columns)
df.shape

In [ ]:
# Represent the column names of the DataFrame

df.columns

In [ ]:

df.dtypes

## 3. Missing Value Analysis

Missing values are identified across all columns to determine which fields require further investigation and appropriate handling.

In [ ]:
df.isnull().sum()
df.duplicated().sum()
df.isnull().sum().sum()

### Investigating Missing Postal Code Values

The Postal Code column contains a large number of missing values. The missing records are investigated by examining their country and market distribution before deciding how to handle the column.


In [ ]:
# Investigate Postal Code

df[df['Postal Code'].isnull()]['Country'].value_counts().head(20)

In [ ]:
df['Country'].nunique()

In [ ]:
df[df['Postal Code'].isnull()]['Market'].value_counts()


In [ ]:
df.duplicated().sum()


In [ ]:
df[df['Postal Code'].isnull()]['Country'].value_counts().head(20)


In [ ]:
df['Country'].nunique()

In [ ]:
df[df['Postal Code'].isnull()]['Market'].value_counts()

In [ ]:
df.duplicated().sum()


In [ ]:
# tells us which countries actually have postal-code data.
df[df['Postal Code'].notna()]['Country'].value_counts().head(20)

In [ ]:
#see what the actual postal-code
df[df['Postal Code'].notna()]['Postal Code'].head(20)

In [ ]:

df['Postal Code'].describe()

### Dataset Overview

A broader statistical overview is generated to examine the distribution and characteristics of the dataset.

In [ ]:
#broad overview of the dataset
df.describe(include='all').T

### Unique Value Analysis

The number of unique values in each column is examined to better understand categorical and identifier fields.

In [ ]:
# ell us how many unique values each column has.
df.nunique().sort_values()

### Categorical Value Check

Categorical fields are reviewed to ensure that their values appear reasonable and consistent.

In [ ]:
#checks whether categorical values look sensible.
df['Order Priority'].value_counts()

## 4. Data Cleaning

A copy of the original dataset is created so that the raw data remains unchanged while cleaning operations are performed on the copied DataFrame.

In [ ]:
# Create a copy for cleaning
df_clean = df.copy()

### Handling Missing Postal Code Values

The Postal Code column contains 41,296 missing values, representing approximately 80.5% of the dataset.

Since most values in this column are missing and reliable values cannot be inferred for the missing records, the Postal Code column is removed rather than artificially filling the missing values.

In [ ]:
# Remove Postal Code because 80.5% of its values are missing
df_clean = df_clean.drop(columns=['Postal Code'])

### Checking the Cleaned Dataset Structure

The shape of the dataset is checked after removing the Postal Code column.

In [ ]:
df_clean.shape 

### Verifying Missing Values

The cleaned dataset is checked again to confirm that no missing values remain.

In [ ]:
df_clean.isnull().sum()
df_clean.isnull().sum().sum()

## 5. Duplicate Record Check

The cleaned dataset is checked for duplicate rows to ensure that repeated records are not present.

In [ ]:
df_clean.duplicated().sum()

## 6. Data Type Validation

The data types of the cleaned dataset are reviewed to ensure that numerical, categorical, and date fields are represented appropriately.

In [ ]:
df_clean.dtypes

## 7. Numerical Data Validation

Numerical columns are examined for unusual or invalid values such as non-positive sales or quantities, discounts outside the expected range, and negative shipping costs.

In [ ]:
# Check numerical columns for suspicious values
df_clean[['Sales', 'Quantity', 'Discount', 'Profit', 'Shipping Cost']].describe()

In [ ]:
# Check for negative or zero values
print("Sales <= 0:", (df_clean['Sales'] <= 0).sum())
print("Quantity <= 0:", (df_clean['Quantity'] <= 0).sum())
print("Discount < 0:", (df_clean['Discount'] < 0).sum())
print("Discount > 1:", (df_clean['Discount'] > 1).sum())
print("Shipping Cost < 0:", (df_clean['Shipping Cost'] < 0).sum())

## 8. Date Validation

Order Date and Ship Date are compared to ensure that no order was shipped before it was placed.

In [ ]:
# Check whether any order was shipped before it was ordered

invalid_dates = df_clean[df_clean['Ship Date'] < df_clean['Order Date']]

print("Orders shipped before order date:", len(invalid_dates))

### Calculating Shipping Duration

The number of days between the Order Date and Ship Date is calculated and stored in a new Shipping Days column.

In [ ]:
# Calculate shipping time in days

df_clean['Shipping Days'] = (
    df_clean['Ship Date'] - df_clean['Order Date']
).dt.days

df_clean['Shipping Days'].describe()

### Validating Shipping Duration

The Shipping Days column is checked to ensure that no negative shipping durations exist.

In [ ]:
# Check for negative shipping duration

print("Negative shipping days:", (df_clean['Shipping Days'] < 0).sum())


## 9. Text Data Cleaning

Text columns are checked for unnecessary leading or trailing whitespace that could cause inconsistencies during analysis.

In [ ]:

# Check for leading/trailing whitespace in text columns

text_columns = df_clean.select_dtypes(include='object').columns

for col in text_columns:
    whitespace_count = (
        df_clean[col].astype(str) != df_clean[col].astype(str).str.strip()
    ).sum()
    
    print(f"{col}: {whitespace_count} values with extra whitespace")

### Checking for Blank Text Values

Text columns are checked for empty or blank values after removing surrounding whitespace.

In [ ]:
# Check for empty or blank text values

for col in text_columns:
    blank_count = df_clean[col].astype(str).str.strip().eq('').sum()
    print(f"{col}: {blank_count} blank values")

### Removing Unnecessary Whitespace

Leading and trailing whitespace is removed from text columns to improve consistency and data quality.

In [ ]:
# Remove leading and trailing whitespace from all text columns

for col in text_columns:
    df_clean[col] = df_clean[col].str.strip()

print("Whitespace cleaning completed.")

### Verifying Text Cleaning

The text columns are checked again to confirm that unnecessary leading or trailing whitespace has been removed.

In [ ]:
# Verify that no extra whitespace remains

for col in text_columns:
    whitespace_count = (
        df_clean[col] != df_clean[col].str.strip()
    ).sum()
    
    print(f"{col}: {whitespace_count} values with extra whitespace")

## 10. Final Data Quality Check

The final dataset is validated to confirm the results of the cleaning and preparation process.


In [ ]:
# Final data quality check

print("Rows:", df_clean.shape[0])
print("Columns:", df_clean.shape[1])
print("Missing values:", df_clean.isnull().sum().sum())
print("Duplicate rows:", df_clean.duplicated().sum())
print("Negative shipping days:", (df_clean['Shipping Days'] < 0).sum())

## 11. Export the Cleaned Dataset

The cleaned and validated dataset is exported as a CSV file for future analysis and visualization.


In [ ]:
# Export the cleaned dataset

output_path = "../data/cleaned/Global_Superstore_Cleaned.csv"

df_clean.to_csv(output_path, index=False)

print("Cleaned dataset exported successfully!")
print(f"Saved to: {output_path}")

## Conclusion

The Global Superstore dataset was successfully cleaned and prepared using Python and Pandas.

### Key outcomes:
- Loaded and inspected the dataset.
- Identified and investigated missing values.
- Removed the Postal Code column due to its high percentage of missing values.
- Checked for duplicate records.
- Validated data types and numerical values.
- Validated order and shipping dates.
- Created a Shipping Days column.
- Cleaned unnecessary whitespace from text fields.
- Performed final data quality validation.
- Exported the cleaned dataset as a CSV file.

The final dataset contains 51,290 rows and 24 columns with no missing values, duplicate rows, or negative shipping durations.